In [3]:
import numpy as np
from scipy.special import betaln, gammaln
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression

# ----------------------------
# Numerics helpers
# ----------------------------
def _clip01(x, eps=1e-6):
    return np.clip(x, eps, 1.0 - eps)

def _logit(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p) - np.log(1 - p)

def _betabinom_logpmf(k, T, a, b):
    # k, a, b can be arrays broadcastable to same shape
    logC = gammaln(T + 1) - gammaln(k + 1) - gammaln(T - k + 1)
    return logC + betaln(k + a, (T - k) + b) - betaln(a, b)

def estimate_mu_per_class(X, y, a0=0.5, b0=0.5):
    y = y.astype(int)
    idx0, idx1 = y == 0, y == 1
    s0, s1 = X[idx0].sum(0), X[idx1].sum(0)
    n0, n1 = max(idx0.sum(), 1), max(idx1.sum(), 1)
    mu0 = (s0 + a0) / (n0 + a0 + b0)
    mu1 = (s1 + a0) / (n1 + a0 + b0)
    return _clip01(mu0), _clip01(mu1)

def estimate_shared_kappa(X, y, mu0, mu1):
    # very rough moment match; keep as you had it
    v0 = X[y == 0].var(0, ddof=1)
    v1 = X[y == 1].var(0, ddof=1)
    num = 0.5 * (mu0*(1-mu0) + mu1*(1-mu1))
    den = 0.5 * (v0 + v1)
    mask = den > 0
    if mask.sum() == 0:
        return 50.0
    kappa = np.mean(num[mask] / den[mask]) - 1
    return float(np.clip(kappa, 2, 1e6))

# ----------------------------
# Interpolating model
# ----------------------------
class BetaBinomEvidencePairwiseLR:
    """
    Interpolates between:
      - Beta–Binomial Naive Bayes evidence (η=0 with only additive score)
      - A coupled model via quadratic interactions in a low-d evidence subspace (η>0)

    Pipeline:
      1) Fit Beta–Binomial params per neuron (a0,b0,a1,b1)
      2) Transform X -> Phi (N x p) evidence
      3) Reduce Phi -> Z (N x d) via TruncatedSVD (randomized)
      4) Build features [Z, eta * vec(triu(Z Z^T))] and fit logistic regression
    """

    def __init__(self, T=10, d=25, eta=0.5, C=1.0, random_state=0):
        self.T = int(T)
        self.d = int(d)
        self.eta = float(eta)
        self.C = float(C)
        self.random_state = int(random_state)

    def fit(self, X, y):
        y = y.astype(int)

        # --- Fit Beta–Binomial marginals
        self.pi_ = float(y.mean())
        mu0, mu1 = estimate_mu_per_class(X, y)
        self.kappa_ = estimate_shared_kappa(X, y, mu0, mu1)
        self.a0_, self.b0_ = mu0*self.kappa_, (1-mu0)*self.kappa_
        self.a1_, self.b1_ = mu1*self.kappa_, (1-mu1)*self.kappa_

        # --- Evidence transform on training data
        Phi = self._evidence(X)  # (N,p)

        # --- Standardize evidence (important for SVD + quadratic stability)
        self.phi_scaler_ = StandardScaler(with_mean=True, with_std=True)
        Phi_s = self.phi_scaler_.fit_transform(Phi)

        # --- Low-d projection (handles huge p)
        self.proj_ = TruncatedSVD(n_components=self.d, random_state=self.random_state)
        Z = self.proj_.fit_transform(Phi_s)  # (N,d)

        # --- Build features and fit logistic regression
        F = self._features_from_Z(Z)  # (N, d + d(d+1)/2)
        self.clf_ = LogisticRegression(
            penalty="l2",
            C=self.C,
            solver="lbfgs",
            max_iter=2000
        )
        self.clf_.fit(F, y)
        return self

    def _evidence(self, X):
        # Phi_ij = log p(x_ij|1) - log p(x_ij|0), using rounded counts
        k = np.rint(self.T * X).astype(int)
        ll1 = _betabinom_logpmf(k, self.T, self.a1_, self.b1_)
        ll0 = _betabinom_logpmf(k, self.T, self.a0_, self.b0_)
        return ll1 - ll0

    def _features_from_Z(self, Z):
        # linear part
        lin = Z

        # quadratic part: upper triangle of outer products
        # For each sample: vec(triu(Z_i Z_i^T))
        N, d = Z.shape
        iu = np.triu_indices(d)
        quad = np.empty((N, len(iu[0])), dtype=Z.dtype)
        for i in range(N):
            G = np.outer(Z[i], Z[i])
            quad[i] = G[iu]

        # scale quadratic strength by eta
        quad *= self.eta

        return np.concatenate([lin, quad], axis=1)

    def predict_proba(self, X):
        Phi = self._evidence(X)
        Phi_s = self.phi_scaler_.transform(Phi)
        Z = self.proj_.transform(Phi_s)
        F = self._features_from_Z(Z)
        return self.clf_.predict_proba(F)

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    def score(self, X):
        # signed logit score if you want it
        proba = self.predict_proba(X)[:, 1]
        return _logit(proba)

from sklearn.model_selection import LeaveOneOut

def loo_accuracy_interpolating(X, y, T=10, d=25, eta=0.5, C=1.0):
    loo = LeaveOneOut()
    preds = np.zeros_like(y, dtype=int)

    for tr, te in loo.split(X):
        model = BetaBinomEvidencePairwiseLR(T=T, d=d, eta=eta, C=C, random_state=0)
        model.fit(X[tr], y[tr])
        preds[te[0]] = model.predict(X[te])[0]

    return (preds == y).mean(), preds

import numpy as np
from sklearn.model_selection import LeaveOneOut
from joblib import Parallel, delayed


vit = np.load('/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl', allow_pickle=True)['natural_scenes']
X   = np.load('/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy').T
y   = (np.argmax(vit,1)<=397).astype(int)

def _loo_one_fold(tr, te, X, y, T, d, eta, C, seed):
    model = BetaBinomEvidencePairwiseLR(
        T=T, d=d, eta=eta, C=C, random_state=seed
    )
    model.fit(X[tr], y[tr])
    return te[0], model.predict(X[te])[0]

def loo_accuracy_interpolating_parallel(
    X, y, T=10, d=25, eta=0.5, C=1.0, n_jobs=12
):
    loo = LeaveOneOut()
    N = len(y)
    preds = np.zeros(N, dtype=int)

    jobs = (
        delayed(_loo_one_fold)(tr, te, X, y, T, d, eta, C, 0)
        for tr, te in loo.split(X)
    )

    results = Parallel(n_jobs=n_jobs, prefer="processes")(jobs)

    for idx, pred in results:
        preds[idx] = pred

    acc = (preds == y).mean()
    return acc, preds


acc0, _ = loo_accuracy_interpolating_parallel(
    X, y, d=25, eta=0.0, C=1.0, n_jobs=12
)

acc1, _ = loo_accuracy_interpolating_parallel(
    X, y, d=25, eta=0.5, C=1.0, n_jobs=12
)

acc2, _ = loo_accuracy_interpolating_parallel(
    X, y, d=25, eta=1.0, C=1.0, n_jobs=12
)

print("eta=0.0:", acc0)
print("eta=0.5:", acc1)
print("eta=1.0:", acc2)

'''
# Example:
acc0, _ = loo_accuracy_interpolating(X, y, d=25, eta=0.0, C=1.0)  # ~ additive-only (no quadratic)
acc1, _ = loo_accuracy_interpolating(X, y, d=25, eta=0.5, C=1.0)  # coupled
acc2, _ = loo_accuracy_interpolating(X, y, d=25, eta=1.0, C=1.0)  # more coupled

print("eta=0.0:", acc0)
print("eta=0.5:", acc1)
print("eta=1.0:", acc2)'''


eta=0.0: 0.6779661016949152
eta=0.5: 0.5932203389830508
eta=1.0: 0.5932203389830508


'\n# Example:\nacc0, _ = loo_accuracy_interpolating(X, y, d=25, eta=0.0, C=1.0)  # ~ additive-only (no quadratic)\nacc1, _ = loo_accuracy_interpolating(X, y, d=25, eta=0.5, C=1.0)  # coupled\nacc2, _ = loo_accuracy_interpolating(X, y, d=25, eta=1.0, C=1.0)  # more coupled\n\nprint("eta=0.0:", acc0)\nprint("eta=0.5:", acc1)\nprint("eta=1.0:", acc2)'